# IL3.4: Escalabilidad y Sostenibilidad
## Notebook 4: Sostenibilidad e IA: Green AI en Agentes de IA

### Objetivo:
Comprender los principios de Green AI y Green Computing aplicados al desarrollo y ejecución de agentes LLM, analizando cómo medir la huella de carbono y cómo diferir procesamiento complejo según la intensidad energética de la red.

### Green AI y la Huella de Carbono
La ejecución y entrenamiento de modelos de lenguaje consumen gigavatios de electricidad. Las prácticas de **Green AI** buscan:
- **Eficiencia primero:** Optimizar algoritmos y prompts antes de simplemente escalar el hardware.
- **Carbon awareness (Sensibilidad al carbono):** Planificar las ejecuciones pesadas para momentos del día o regiones de data centers donde la matriz eléctrica tenga una alta proporción de fuentes limpias (solar, eólica).
- **Minimización de recursos:** Utilizar técnicas de compresión del modelo (cuantización de peso, destilación) para correr modelos optimizados.


### Inicialización del Entorno
Ejecuta la siguiente celda para configurar el cliente LLM y el agente real de Wikipedia usando LangChain.


In [ ]:
import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_base=os.environ.get("GITHUB_BASE_URL"),
        openai_api_key=os.environ.get("GITHUB_TOKEN"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

from langchain_classic.agents import tool, create_openai_tools_agent, AgentExecutor
from langsmith import Client

@tool
def get_wikipedia_summary(query: str) -> str:
    """Busca en Wikipedia un tema y devuelve un resumen de 2 frases. Útil para obtener información sobre personas, lugares o conceptos."""
    try:
        return wikipedia.summary(query, sentences=2)
    except Exception as e:
        return f"Ocurrió un error: {e}"

tools = [get_wikipedia_summary]

client = Client(None)
# Usamos el mismo prompt de la comunidad que ya está preparado para manejar historial
prompt = client.pull_prompt("hwchase17/openai-tools-agent")

agent = create_openai_tools_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

print("✅ Agente y herramientas listos.")


### Agente Sostenible Sensible al Carbono
Implementaremos un agente (`SustainableAgent`) que utiliza un estimador de la intensidad actual de la red de energía. Si la intensidad supera un umbral permitido, difiere las consultas no críticas encolándolas para horarios nocturnos u horas verdes, protegiendo el medio ambiente.


In [ ]:
import time
import random

class CarbonTracker:
    def get_current_intensity(self):
        # Retorna intensidad de carbono simulada en gCO2/kWh
        # Limpio: < 150, Moderado: 150-250, Sucio: > 250
        return random.randint(50, 400)

class EnergyOptimizer:
    def compress_model(self):
        return "Modelo comprimido (Cuantización de peso de FP32 a INT8). Consumo de energía -60%."

class SustainableAgent:
    def __init__(self, executor, threshold=200):
        self.executor = executor
        self.carbon_tracker = CarbonTracker()
        self.energy_optimizer = EnergyOptimizer()
        self.threshold = threshold

    def process_with_carbon_awareness(self, request, critical=False):
        carbon_intensity = self.carbon_tracker.get_current_intensity()
        print(f"[Monitoreo Verde] Intensidad actual de carbono en la red: {carbon_intensity} gCO2/kWh")
        
        # Si la red está muy cargada o sucia y el requerimiento no es de alta prioridad
        if carbon_intensity > self.threshold and not critical:
            print(f"[AVISO COMPLIANCE] Intensidad de carbono excede el límite permitido ({carbon_intensity} > {self.threshold}).")
            return self.schedule_for_low_carbon_time(request)
        
        # Procesar inmediatamente
        return self.process_immediately(request)

    def process_immediately(self, request):
        print("[Ejecución] Procesando consulta inmediatamente...")
        try:
            if llm is None:
                return "Resultado simulado bajo carbono."
            res = self.executor.invoke({"input": request})
            return res.get("output", "")
        except Exception as e:
            return f"Error: {e}"

    def schedule_for_low_carbon_time(self, request):
        return f"[ENCOLADO] Consulta '{request}' encolada para ejecución en horas de baja huella de carbono."

# Probar agente sostenible
agent_sust = SustainableAgent(agent_executor, threshold=200)

print("--- Ejecución 1: Solicitud No Crítica (Seguimiento ambiental) ---")
print(agent_sust.process_with_carbon_awareness("Buscar biografía de Isaac Newton", critical=False))

print("\n--- Ejecución 2: Solicitud Crítica (Ejecución inmediata obligada) ---")
print(agent_sust.process_with_carbon_awareness("Buscar biografía de Isaac Newton", critical=True))


### Métricas de Sostenibilidad en IA
Para medir la huella energética de nuestra aplicación, evaluamos:
- **Wh por consulta:** Consumo eléctrico promedio por interacción.
- **Matriz de Intensidad Energética:** Contribución en gramos de CO2 equivalente por kilovatio hora.
- **Trade-off Precisión vs. Energía:** Analizar si usar modelos alternativos más pequeños de menor consumo genera resultados aceptables para el caso de negocio.


### Preguntas de Análisis
1. **¿Cuáles son los principales factores que incrementan el consumo de energía en una arquitectura de agentes multitarea?**
2. **¿De qué manera la cuantización del modelo de LLM aporta a la sostenibilidad de los servidores locales?**
3. **Si fueras el arquitecto de una solución corporativa de IA, ¿qué estrategias propondrías para conciliar el tiempo de respuesta rápido con el cumplimiento de las metas de sostenibilidad ambiental (Green Computing)?**
